# LlamaIndex Lab: Ingest, Index, and Query Documents

**Goal:** Ingest raw text/documents into LlamaIndex, create a vector index, and run retrieval queries.

**Estimated Time:** ~90 minutes

## 1) Setup
Install required packages (run this once per environment):

In [ ]:
# %pip install llama-index-core llama-index-embeddings-openai llama-index-llms-openai python-dotenv

Create a `.env` file in your project/notebook directory with:

```env
OPENAI_API_KEY=your_api_key_here
```

## 2) Imports and Initialization

In [ ]:
import os
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
# from llama_index.embeddings.openai import OpenAIEmbedding
# from llama_index.llms.openai import OpenAI
from llama_index.embeddings.ollama import OllamaEmbedding
#from llama_index.llms.ollama import Ollama

load_dotenv()

# if not os.getenv("OPENAI_API_KEY"):
#     raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")

# llm = OpenAI(model="gpt-4o-mini", temperature=0)
# embed_model = OpenAIEmbedding(model="text-embedding-3-small")


#llm = Ollama(model="llama3.2:latest", base_url="http://localhost:11434", request_timeout=120.0, temperature=0)
#embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest", base_url="http://localhost:11434")



TypeError: unsupported operand type(s) for |: 'type' and 'type'

## 3) Prepare Data
Create a small dataset with one file each for LangChain, LangGraph, and CrewAI.

In [ ]:
os.makedirs("data", exist_ok=True)

docs = {
    "langchain.txt": """LangChain is a framework that helps developers build applications powered by LLMs with tools, memory, and chaining capabilities.""",
    "langgraph.txt": """LangGraph is a framework for building stateful, graph-based workflows for AI agents with explicit nodes and edges.""",
    "crewai.txt": """CrewAI is a framework for orchestrating multi-agent collaboration, assigning roles and tasks across multiple AI workers."""
}

for name, text in docs.items():
    with open(f"data/{name}", "w", encoding="utf-8") as f:
        f.write(text)

print("Wrote sample files:", ", ".join(docs.keys()))

## 4) Load Documents with LlamaIndex

In [ ]:
reader = SimpleDirectoryReader("data")
documents = reader.load_data()

print(f"Loaded {len(documents)} documents")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc.metadata.get('file_name', 'unknown')}")

## 5) Build a Vector Index

In [ ]:
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)
query_engine = index.as_query_engine(llm=llm)

print("Index built successfully.")

## 6) Run Sample Queries

In [ ]:
queries = [
    "What is LangChain used for?",
    "Which framework uses nodes and edges for workflows?",
    "Which framework supports multi-agent role assignment?"
]

for q in queries:
    print(f"\nQ: {q}")
    print("A:", query_engine.query(q).response)

## 7) Add New Data Dynamically

In [ ]:
new_doc = """AutoGPT is an experimental framework that loops planning, acting, and reflecting steps to achieve open-ended goals."""
index.insert(Document(text=new_doc, metadata={"source": "autogpt"}))

question = "Which framework uses planning, acting, and reflecting loops?"
print(f"\nQ: {question}")
print("A:", query_engine.query(question).response)

## 8) Interactive Query Cell
Run this cell multiple times and type your own question each time.

In [ ]:
user_q = input("Ask a question about the loaded docs: ")
if user_q.strip():
    print("A:", query_engine.query(user_q).response)
else:
    print("Please enter a non-empty question.")

## 9) Persist and Reload the Index

In [ ]:
# Persist to disk
index.storage_context.persist(persist_dir="storage")
print("Persisted index to ./storage")

In [ ]:
from llama_index.core import load_index_from_storage, StorageContext

storage = StorageContext.from_defaults(persist_dir="storage")
reloaded_index = load_index_from_storage(storage, embed_model=embed_model)
reloaded_engine = reloaded_index.as_query_engine(llm=llm)

print(reloaded_engine.query("What does CrewAI help orchestrate?").response)

## 10) Extensions
- Swap `SimpleDirectoryReader` for other loaders
  - Web pages (`download_loader("SimpleWebPageReader")`)
  - PDFs, Notion, Slack, etc. via LlamaHub integrations
- Add metadata (e.g., source/date/author) and filter at retrieval time
- Experiment with chunking strategies and retrieval settings

## 11) What You Learned
✅ How to ingest docs into LlamaIndex  
✅ How to build and query a `VectorStoreIndex`  
✅ How to add docs dynamically and persist/reload an index  
✅ How this becomes the data layer for agentic AI workflows

Great job — this is the foundation for building retrieval-enabled AI agents.